# 5. Neural surrogate for the ray-tracing map

Rendering by brute force costs one geodesic integration per pixel. Instead, we learn the map

$$(b_y, b_z, \theta_{obs}) \longmapsto (r_{hit}, \phi_{hit})$$

with a small MLP, then apply $T(r)$ and the Doppler factor in post-processing. A full image becomes one batched forward pass, so the viewing angle can be changed interactively.

Generate the dataset first (takes a while; uses all cores):

```
python scripts/generate_dataset.py --n-angles 20 --n-per-angle 100 --k-max 3
```

Each row stores up to `k_max` equatorial crossings ($n=0$ direct image, $n=1$ first lensed image, ...) and extra samples are placed just outside the critical impact parameter, where the lensed ring lives.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from geodesics import Schwarzschild, temperature_interpolator, show_image
from geodesics.surrogate import load_dataset, build_model, train, evaluate, render_nn

## Look at the data

In [ ]:
data = np.load('../raytracing_dataset_Schwarzschild.npy')
print(data.shape)
th = data[:, 2]
th0 = th[np.argmin(np.abs(th - 12))]
d = data[np.isclose(th, th0)]
fig, ax = plt.subplots(1, 2, figsize=(12, 6))
for n, a in enumerate(ax):
    hit = d[:, 4 + 3*n + 2] > 0
    a.scatter(d[hit, 0], d[hit, 1], s=2)
    a.set_aspect('equal'); a.set_title(f'pixels with image n={n}, theta={th0:.0f}°')
plt.show()

## Train on the direct image ($n=0$), disk hits only

In [ ]:
loader, X_test, Y_test, norm = load_dataset('../raytracing_dataset_Schwarzschild.npy', hits_only=True, order=0)
model = build_model()
history = train(model, loader, X_test, Y_test, norm, epochs=1000, lr=2e-4, log_every=50)

In [ ]:
evaluate(model, X_test, Y_test, norm)

In [ ]:
# torch.save(model.state_dict(), '../raytracing_model_Schwarzschild.pt')

## Render with the network

In [ ]:
schw = Schwarzschild(M_val=1.0)
T_interp = temperature_interpolator(schw, r_max=30)
for angle in (5, 17, 45):
    img = render_nn(model, norm, schw, T_interp, theta_obs_deg=angle, N=600, b_max=12)
    show_image(img, b_max=12, title=f'NN render, {angle}°')

## Does `k_max > 1` capture the ring?

Ground truth from the parallel tracer, counting only the first lensed image.

In [ ]:
from geodesics.dataset import render_truth
img1 = render_truth(12, N=200, k_max=1)
img3 = render_truth(12, N=200, k_max=3, orders=[1])
fig, ax = plt.subplots(1, 2, figsize=(14, 7))
for a, im, t in [(ax[0], img1, 'direct image only'), (ax[1], img3, 'first lensed image (n=1)')]:
    a.imshow(im, cmap='inferno', origin='lower', extent=[-12, 12, -12, 12]); a.set_title(t); a.axis('off')
plt.tight_layout(); plt.show()